<h1>Logit Lens and Tuned Logit Lens</h1>
<b>Stone Amsbaugh - MATH498 Large Language Models - Spring 2026</b>

The code in this notebook will demonstrate how to utilize the logit lens utility functions defined in logit_lens.py, in order to make your own lens and use it for naive interpretability.

In [8]:
from logit_lens import Lens, Translator

First, lets load our sample language model, which is EleutherAI/pythia-14m by default.

In [4]:
model = Lens.get_model()

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-14m into HookedTransformer


Now we can initialize a plain lens object. It takes a model, a query for the model, and optionally the top k tokens to visualize.

As a brief overview, this lens object is essentially just looking at the vector at each layer, and treating them as logits of the probability distribution that our output layer is taken to be. Ideally, it will let us see where our model is at in the intermediate layers, not just in the one it ultimately lets us see.

In [5]:
lens = Lens(model, "The sky is blue and the grass is", top_k = 5)

SUCCESS: Lens object created.


Now we can visualize the lens with the to_table method. We will set HTML to True to see the heatmap, whereas the default markdown table format won't be colored.

In [6]:
lens.to_table(html=True)

Rank,Layer 0,Layer 1,Layer 2,Layer 3,Layer 4,Layer 5
1,'люч'15.8,'itle'16.1,' increased'13.7,"' ""'17.8",' red'16.6,' blue'15.1
2,'од'15.5,' present'13.5,'đ'13.3,' the'17.3,' a'16.0,' red'14.8
3,'itle'14.3,'sworth'13.5,' being'13.3,"'""'17.3",' true'15.8,' green'14.7
4,'еп'14.1,'itious'13.4,' improved'13.0,' red'17.2,' new'15.7,' yellow'14.0
5,'ож'14.1,' satisfactory'13.4,'oplan'12.7,' a'17.1,' F'15.2,' white'13.9


Notice how by the time we get to the final layer, the top tokens by logit are colors, which makes sense given the prompt.
However, this isn't really fair. The layers besides the final layer don't make a lot of sense, and they have never had any reason to directly map to the same output vector as our final layer, and instead may represent intermediate information differently.

We can attempt to address this issue by using a 'tuned' logit lens. We can think about training a neural network to map these intermediate layers to the result of the output layer. This allows us to get a better idea of what words in our vocabulary these intermediate thoughts are actually indicating. 

To implement this, we will provide our lens with another parameter, a Translator object. But first, we need to create this object and train it on some text.

In [ ]:
translator = Translator(model)

Translator initialised. Call .train() or .load() before use.


Now we will train the model, and save it to a file, in case we ever want to load it again in the future.

If you would like to use a pre-trained translator, or just get the same results as me, you can download my [translator here](https://drive.google.com/file/d/1sgD1pppIYIrBEeSYSuK1XDFImrXby5C9/view?usp=sharing) and load it using the code in the comments in thefollowing cell.

In [ ]:
translator.train(save_path="translator.pt")
# OR uncomment the following line to load the translator from a file.
# translator = Translator.from_file('translator.pt', model)

Translator initialised. Call .train() or .load() before use.
Translator loaded from translator.pt


Now that we have a trained translator object, we can hook it up to a lens and see the results.

In [13]:
tuned_lens = Lens(model, "The sky is blue and the grass is", translator=translator, top_k=5)

SUCCESS: Lens object created.


Visualize it the same way as we did previously

In [14]:
tuned_lens.to_table(html=True)

Rank,Layer 0,Layer 1,Layer 2,Layer 3,Layer 4,Layer 5
1,' a'14.2,' a'15.3,' a'14.1,' blue'15.9,' red'15.1,' blue'15.1
2,' the'13.8,' the'14.6,' the'13.5,' red'15.4,' blue'15.0,' red'14.7
3,' not'12.8,' an'14.1,' much'13.5,' white'14.7,' green'14.1,' green'14.7
4,' an'12.7,' currently'14.0,' improved'13.3,' a'14.7,' white'13.8,' yellow'14.0
5,' present'11.9,' being'13.7,' also'13.3,' green'14.4,' black'13.5,' white'13.9


We can see this working better already. Whereas before some of the intermediate layers were non-english characters ro otherwise nonsensical, we can now see colors as early as later 3, and perhaps the indication of some other so-called 'thoughts' before that. It is almost as if it is branching out from more generic words and getting it more specific as we think or propagate more through the transformers.

However, is this really interpretability? If someone were to ask you what this model, which is still orders of magnitude simpler than cutting-edge language models, could you really say anything about what is going on under the hood that isn't primarily a guess based on very little and very fuzzy information? I certainly couldn't say very much about the LLMs thought processes at all.

And this is for a very simple question about colors. Imagine we are trying to look at bigger questions, like the following example:

In [41]:
lens3 = Lens(model, 
        "I am going to take over the", translator=translator)
lens3.to_table()

SUCCESS: Lens object created.


| Rank | Layer 0 | Layer 1 | Layer 2 | Layer 3 | Layer 4 | Layer 5 |
| --- | --- | --- | --- | --- | --- | --- |
| **1** | `' United'` 11.5 | `' first'` 11.3 | `'\n'` 11.1 | `'\n'` 11.4 | `'\n'` 10.4 | `' first'` 10.8 |
| **2** | `' following'` 11.4 | `' United'` 10.9 | `' "'` 10.6 | `' "'` 10.4 | `' "'` 9.6 | `' right'` 10.7 |
| **3** | `' same'` 11.3 | `' problem'` 10.8 | `'Q'` 10.5 | `' question'` 9.5 | `' world'` 9.3 | `' next'` 10.4 |
| **4** | `' role'` 11.2 | `' world'` 10.6 | `' world'` 10.2 | `'Q'` 9.5 | `' problem'` 9.3 | `' world'` 10.3 |
| **5** | `' user'` 11.0 | `' same'` 10.6 | `' question'` 9.8 | `' He'` 9.4 | `' children'` 9.0 | `' people'` 10.3 |

Even from Layer 1, we can see the following token of 'word' appearing towards the top of the models options based on logits. So is my LLM misaligned? Does it want to take over the world? The problem is, we really don't know, this approach is just too simplistic.

One of the primary reasons for this is that thoughts aren't expressed as one word, or even a probability distribution over all words. We express thoughts using sentences, which isn't compatible with this current approach.

However, this is useful as a first example into how we can work with LLMs to open them up, and what kind of information is available to us. 
